In [ ]:
%%writefile combined.cu

#include <iostream>
using namespace std;

// Kernel for Addition
__global__ void add(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if(i < n)
        c[i] = a[i] + b[i];
}

// Kernel for Multiplication
__global__ void multiply(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if(i < n)
        c[i] = a[i] * b[i];
}

int main() {

    const int n = 1000;

    int *a = new int[n];
    int *b = new int[n];

    int *c_add = new int[n];
    int *c_mul = new int[n];

    // Initialize arrays
    for(int i = 0; i < n; i++) {
        a[i] = i;
        b[i] = i + 1;
    }

    int *da, *db, *dc_add, *dc_mul;

    int size = n * sizeof(int);

    // Allocate GPU memory
    cudaMalloc(&da, size);
    cudaMalloc(&db, size);
    cudaMalloc(&dc_add, size);
    cudaMalloc(&dc_mul, size);

    // Copy data to GPU
    cudaMemcpy(da, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(db, b, size, cudaMemcpyHostToDevice);

    // Define threads & blocks
    int threads = 256;
    int blocks = (n + threads - 1) / threads;

    // Launch kernels
    add<<<blocks, threads>>>(da, db, dc_add, n);
    multiply<<<blocks, threads>>>(da, db, dc_mul, n);

    // Copy results back
    cudaMemcpy(c_add, dc_add, size, cudaMemcpyDeviceToHost);
    cudaMemcpy(c_mul, dc_mul, size, cudaMemcpyDeviceToHost);

    // Print first 10 results
    cout << "Addition (first 10): ";
    for(int i = 0; i < 10; i++)
        cout << c_add[i] << " ";

    cout << endl;

    cout << "Multiplication (first 10): ";
    for(int i = 0; i < 10; i++)
        cout << c_mul[i] << " ";

    cout << endl;

    // Free GPU memory
    cudaFree(da);
    cudaFree(db);
    cudaFree(dc_add);
    cudaFree(dc_mul);

    // Free CPU memory
    delete[] a;
    delete[] b;
    delete[] c_add;
    delete[] c_mul;

    return 0;
}


convert runtime to GPU

1.compile
!nvcc co.cu -o co

2.run
!./co